## Setup

In [1]:
import json
import os
from pathlib import Path

import pandas as pd

In [2]:
HOME_DIRECTORY = Path('..')
DATASET_PATH = HOME_DIRECTORY / "data" / "dataset"

In [3]:
# Hugging Face token needed for some gated datasets
try: 
    with open(HOME_DIRECTORY / 'config.json') as configfile:
        cfg = json.load(configfile)
        os.environ['HF_TOKEN'] = cfg['HF_TOKEN']
    print('Token found')
    HAS_TOKEN = True
except:
    print('Token NOT FOUND')
    HAS_TOKEN = False

Token found


In [4]:
# Define our datasets

# Dataset 1: 
dataset_1_name = "AzharAli05_Resume_Screening_Dataset"
dataset_1_url = "hf://datasets/AzharAli05/Resume-Screening-Dataset/dataset.csv"

# Dataset 2: Split into train/test.
dataset_2_name = "cnamuangtoun_resume_job_description_fit"
dataset_2_url_1 = "hf://datasets/cnamuangtoun/resume-job-description-fit/train.csv"
dataset_2_url_2 = "hf://datasets/cnamuangtoun/resume-job-description-fit/test.csv"

# Dataset 3: 
dataset_3_name = "MikePfunk28_resume_training_dataset"
dataset_3_url = "hf://datasets/MikePfunk28/resume-training-dataset/training_data.jsonl"

# Other datasets:
# datasetmaster/resumes: "hf://dataset/datasetmaster/resumes/master_resumes.jsonl"

## Dataset Prep

### Dataset 1
https://huggingface.co/datasets/AzharAli05/Resume-Screening-Dataset

In [5]:
df_1_raw = pd.read_csv(dataset_1_url)
print(df_1_raw.shape)
df_1_raw.head()

(10174, 5)


,Role,Resume,Decision,Reason_for_decision,Job_Description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,Game Developer,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...


In [6]:
# I'll process the whole dataset down to a subset with a known format
# we don't mind losing a bunch of resumes, we don't need that many
df_1 = df_1_raw.copy()
df_1.columns = ['role', 'resume', 'decision', 'reasoning', 'description']
df_1['decision'] = (df_1['decision'] == "select")
print(df_1.shape)
df_1.head()

(10174, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,Game Developer,Here's a professional resume for Ann Marshall:...,True,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...


In [7]:
# Only extract resumes with a known format
# "Here" as a prefix indicates that this a LLM-generated resume ("Here's a professional resume for...")
# while "Available upon request" indicates that this resume has a references section, and so we know where it ends
prefix = "Here"
suffix = "Available upon request."
df_1 = df_1[df_1['resume'].str.startswith(prefix)]
df_1 = df_1[df_1['resume'].str.contains(suffix)]
print(df_1.shape)
df_1.head()

(6067, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
5,Mobile App Developer,"Here's a sample resume for Jose Hall, a skille...",False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...


In [8]:
# Remove the "Here's a resume!" prefix
df_1['resume'] = df_1['resume'].str.split(':\n\n', n=1).str[1]
df_1 = df_1[~df_1['resume'].isna()]
print(df_1.shape)
df_1.head()

(6067, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Jason Jones\nE-commerce Specialist\n\nContact ...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
2,Human Resources Specialist,Patrick Mcclain\nHuman Resources Specialist\n\...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Patricia Gray\nContact Information:\n\n* Email...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Amanda Gross\nContact Information:\n\n* Email:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
5,Mobile App Developer,Jose Hall\nContact Information:\n\n* Email: [j...,False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...


In [9]:
# and remove anything after the references (this is usually LLM commentary)
df_1['resume'] = df_1['resume'].str.rsplit(suffix, n=1).str[0] + suffix
print(df_1.shape)
df_1.head()

(6067, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Jason Jones\nE-commerce Specialist\n\nContact ...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
2,Human Resources Specialist,Patrick Mcclain\nHuman Resources Specialist\n\...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Patricia Gray\nContact Information:\n\n* Email...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Amanda Gross\nContact Information:\n\n* Email:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
5,Mobile App Developer,Jose Hall\nContact Information:\n\n* Email: [j...,False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...


In [10]:
# Explore: how many unique positions are there?
df_1_resume_counts = df_1.groupby('description')['resume'].count().reset_index()
df_1_resume_counts.columns = ['description', 'resume_count']
print(df_1_resume_counts.shape)
df_1_resume_counts

(1714, 2)


,description,resume_count
0,**Job Title: Data Scientist**\n\n**Job Summary...,1
1,**Job Title: Database Administrator**\n\n**Job...,1
2,**Job Title: HR Specialist**\n\n**Job Summary:...,1
3,**Job Title: UI/UX Designer**\n\n**Job Summary...,1
4,**Job Title:** AI Engineer\n\n**Job Summary:**...,1
...,...,...
1709,We're seeking a talented UI Engineer to work o...,7
1710,We're seeking a talented UI Engineer to work o...,5
1711,We're seeking a talented UX Designer to work o...,5
1712,We're seeking a talented UX Designer to work o...,5


In [11]:
# and of these positions, how many have both accepted and rejected resumes?
df_1_dual_decision = df_1.groupby('description')['decision'].nunique().reset_index()
df_1_dual_decision.columns = ['description', 'decision_types']
df_1_dual_decision = pd.merge(
    df_1_dual_decision, df_1_resume_counts,
    on='description', how='left'
)
df_1_dual_decision = df_1_dual_decision[df_1_dual_decision['decision_types'] == 2]
print(df_1_dual_decision.shape)
df_1_dual_decision

(864, 3)


,description,decision_types,resume_count
66,"As a AI Researcher, you will play a pivotal ro...",2,8
67,"As a AI Researcher, you will play a pivotal ro...",2,6
68,"As a AI Researcher, you'll lead the design and...",2,4
69,"As a AI Researcher, you'll lead the design and...",2,5
70,"As a AI Researcher, you'll lead the design and...",2,4
...,...,...,...
1708,We're seeking a talented UI Engineer to work o...,2,2
1709,We're seeking a talented UI Engineer to work o...,2,7
1710,We're seeking a talented UI Engineer to work o...,2,5
1712,We're seeking a talented UX Designer to work o...,2,5


In [12]:
# Narrow the dataset down to positions with both accepts and rejects
df_1 = df_1[df_1['description'].isin(df_1_dual_decision['description'])]
df_1 = df_1.reset_index(drop=True)
print(df_1.shape)
df_1.head()

(4861, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Jason Jones\nE-commerce Specialist\n\nContact ...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,E-commerce Specialist,Patricia Gray\nContact Information:\n\n* Email...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
2,E-commerce Specialist,Amanda Gross\nContact Information:\n\n* Email:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
3,Mobile App Developer,Jose Hall\nContact Information:\n\n* Email: [j...,False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...
4,Cloud Engineer,Jessica Hall\nCloud Engineer\n\nContact Inform...,False,Needs improvement in machine learning algorithms.,We're seeking a talented Cloud Engineer to wor...


In [13]:
# Sample a job opening from the entire dataset, and show all its applications
random_state = 0
df_1_subset = df_1[df_1['description'] == df_1['description'].sample(random_state = random_state).item()]
print(df_1_subset['description'].iloc[0])
df_1_subset

As a System Administrator, you will play a pivotal role in shaping the future of e-commerce.


,role,resume,decision,reasoning,description
311,System Administrator,Christina Davis\nSystem Administrator\n\nConta...,False,Needs improvement in machine learning algorithms.,"As a System Administrator, you will play a piv..."
915,System Administrator,Mary Johnson\nSystem Administrator\n\nContact ...,False,Insufficient system design expertise for senio...,"As a System Administrator, you will play a piv..."
2133,System Administrator,John Irwin\nSystem Administrator\n\nContact In...,True,Impressive leadership and communication abilit...,"As a System Administrator, you will play a piv..."
3202,System Administrator,Travis Dawson\nSystem Administrator\n\nContact...,True,Perfectly aligned with data engineering needs.,"As a System Administrator, you will play a piv..."
3827,System Administrator,Marvin Bates\nSystem Administrator\n\nContact ...,True,Excellent full-stack development experience.,"As a System Administrator, you will play a piv..."
4105,System Administrator,Matthew Moreno\nSystem Administrator Candidate...,False,Lacked leadership skills for a senior position.,"As a System Administrator, you will play a piv..."
4258,System Administrator,Jerry Nelson\nSystem Administrator Candidate\n...,True,Strong technical skills in AI and ML.,"As a System Administrator, you will play a piv..."


In [14]:
# and sample a random resume 
print(df_1_subset['resume'].sample().item())

Travis Dawson
System Administrator

Contact Information:

* Email: [travis.dawson@email.com](mailto:travis.dawson@email.com)
* Phone: 555-555-5555
* LinkedIn: linkedin.com/in/travisdawson

Professional Summary:

Highly motivated and experienced System Administrator with a strong background in Windows Server, Networking, and Virtualization. Proven track record of designing, implementing, and maintaining scalable and secure IT infrastructure solutions. Skilled in troubleshooting complex technical issues and collaborating with cross-functional teams to drive business success.

Technical Skills:

* Operating Systems: Windows Server 2012/2016/2019, Windows 10
* Networking: Cisco routers and switches, firewall configuration (Cisco ASA), VPN setup (PPTP/L2TP/IPSec)
* Virtualization: VMware vSphere (vCenter, ESXi), Microsoft Hyper-V
* Cloud Computing: Azure, AWS (EC2, S3)
* Scripting: PowerShell, Batch scripting
* Storage: SAN/NAS management, iSCSI configuration

Professional Experience:

Seni

### Dataset 2

https://huggingface.co/datasets/cnamuangtoun/resume-job-description-fit

In [15]:
df_2_1_raw = pd.read_csv(dataset_2_url_1)
df_2_2_raw = pd.read_csv(dataset_2_url_2)
df_2_raw = pd.concat((df_2_1_raw, df_2_2_raw)).reset_index(drop=True)
df_2_raw.head()

,resume_text,job_description_text,label
0,SummaryHighly motivated Sales Associate with e...,Net2Source Inc. is an award-winning total work...,No Fit
1,Professional SummaryCurrently working with Cat...,At Salas OBrien we tell our clients that were ...,No Fit
2,SummaryI started my construction career in Jun...,Schweitzer Engineering Laboratories (SEL) Infr...,No Fit
3,SummaryCertified Electrical Foremanwith thirte...,"Mizick Miller & Company, Inc. is looking for a...",No Fit
4,SummaryWith extensive experience in business/r...,Life at Capgemini\nCapgemini supports all aspe...,No Fit


In [16]:
df_2 = df_2_raw.copy()
print(df_2['resume_text'].sample().item())

SummaryElectrical Engineer offering more than 4 years of experience in Electrical and Electronics Manufacturing sector,Well-versed in developing technical studies and investigations.Committed job seeker with a history of meeting company needs with consistent and organized practices. Skilled in working under pressure and adapting to new situations and challenges to best enhance the organizational brand.
SkillsElectrical Systems SpecificationsAnalyzing and problem solvingTesting CoordinationProject ManagementData CollectionDiagnosis and troubleshootingMicrosoft office operation
Education and TrainingUniversity of GondarGondar,Amhara Regional State,Ethiopia,Expected in––Bachelor of Science:Electrical and Computer Engineering-GPA:
ExperienceRtx-Electrical EngineerDerry,NH,08/2021-05/2023Provided troubleshooting and analysis for discovered electrical faults, offering potential solutions.Designed, controlled and installed electrical systems and products.Trained and guided new employees accor

### Dataset 3

https://huggingface.co/datasets/MikePfunk28/resume-training-dataset

In [17]:
if HAS_TOKEN:
    df_3_raw = pd.read_json(dataset_3_url, lines=True)
else:
    df_3_raw = pd.DataFrame()
df_3_raw.head()

,messages
0,"[{'role': 'system', 'content': 'You are an exp..."
1,"[{'role': 'system', 'content': 'You are an exp..."
2,"[{'role': 'system', 'content': 'You are an exp..."
3,"[{'role': 'system', 'content': 'You are an exp..."
4,"[{'role': 'system', 'content': 'You are an exp..."


In [18]:
df_3 = df_3_raw.copy()
if HAS_TOKEN:
        df_3 = pd.DataFrame(df_3['messages'].apply(lambda lst: {dct['role']: dct['content'] for dct in lst}).tolist())
print(df_3.shape)
df_3.head()

(22855, 3)


,system,user,assistant
0,You are an expert resume assistant. You help u...,Please summarize the following resume:\n\nGENE...,The candidate is a highly motivated General Ma...
1,You are an expert resume assistant. You help u...,Please summarize the following resume:\n\nDIRE...,The candidate is a seasoned Certified Manageme...
2,You are an expert resume assistant. You help u...,What job category does this resume best fit?\n...,This resume best fits the TEACHER category.
3,You are an expert resume assistant. You help u...,Please summarize the following resume:\n\nStep...,Stephanie Nelson is an experienced AI research...
4,You are an expert resume assistant. You help u...,Please summarize the following resume:\n\nDani...,Danielle Barnes is a results-driven digital ma...


In [19]:
if HAS_TOKEN:
    # System prompt is a bit useless. Drop it.
    unique_sys = df_3['system'].unique()
    print(unique_sys)
    assert len(unique_sys) == 1

    delim = "\n\n"
    df_3[['user_prompt', 'user_resume']] = df_3['user'].str.split(delim, n=1, expand=True)
    df_3 = df_3[['user_resume', 'user_prompt', 'assistant']] # dropped system prompt
print(df_3.shape)
df_3.head()

['You are an expert resume assistant. You help users write, critique, and improve their resumes to land their dream job.']
(22855, 3)


,user_resume,user_prompt,assistant
0,GENERAL MANAGER/FITNESS DIRECTOR Executive Pro...,Please summarize the following resume:,The candidate is a highly motivated General Ma...
1,DIRECTOR OF FINANCE Summary Seasoned Certified...,Please summarize the following resume:,The candidate is a seasoned Certified Manageme...
2,"TEACHER Summary Accomplished, exper...",What job category does this resume best fit?,This resume best fits the TEACHER category.
3,Stephanie Nelson AI Researcher Contact Informa...,Please summarize the following resume:,Stephanie Nelson is an experienced AI research...
4,Danielle Barnes Contact Information: * Email: ...,Please summarize the following resume:,Danielle Barnes is a results-driven digital ma...


In [20]:
if HAS_TOKEN:
    print(df_3['user_resume'].sample().item())

SALES MANAGER/ TERRITORY SALES MANAGER           Experience      Sales Manager/ Territory Sales manager   02/2014   to   08/2015     Company Name   City  ,   State       Selling and working with Franchises, Strategic Partners on Mobile Loyalty Platform.  Working with Digital and Advertising Agencies on Reselling ProductSelling Local Clients in the Arkansas Territory on the Mobile Loyalty Platform.          Marketing Executive/Senior Sales Consultant   04/2011   to   01/2014     Company Name   City  ,   State       Aggressively research, develop, and cultivate leads for LivingSocial Deals using a variety of online and offline sourcesMeet and strive to exceed individual monthly, quarterly, and annual sales goalsQualify prospective clients by phone and close deals in-personUse consultative sales skills to assess merchant goals, propose a customized LivingSocial solution, and obtain commitmentManage relationships with established clients and construct proposals and contracts within selling

## Final Datasets

In [21]:
# Dataset 1
df_1.sample(10, random_state=0)

,role,resume,decision,reasoning,description
311,System Administrator,Christina Davis\nSystem Administrator\n\nConta...,False,Needs improvement in machine learning algorithms.,"As a System Administrator, you will play a piv..."
154,Product Manager,Anthony Mckenzie\nContact Information:\n\n* Ad...,False,Lacks hands-on experience with cloud platforms.,"As a Product Manager, you will play a pivotal ..."
2027,Product Manager,Sheila Nguyen\nContact Information:\n\n* Addre...,True,Impressive leadership and communication abilit...,Looking for an experienced Product Manager to ...
1687,Cloud Architect,Jonathan Cook\nCloud Architect\n\nContact Info...,False,Insufficient system design expertise for senio...,Be part of a passionate team at the forefront ...
1519,IT Support Specialist,Richard Fuller\nContact Information:\n\n* Phon...,False,Insufficient system design expertise for senio...,We are looking for an experienced IT Support S...
2044,Data Analyst,Katherine Tyler\nContact Information:\n\n* Ema...,True,Strong technical skills in AI and ML.,We are looking for an experienced Data Analyst...
4100,AI Researcher,Crystal Park\nContact Information:\n\n* Email:...,False,Needs improvement in machine learning algorithms.,Be part of a passionate team at the forefront ...
4684,QA Engineer,Raymond Greer\nQA Engineer\n\nContact Informat...,False,No experience in back-end development.,Help us build the next-generation products as ...
1881,DevOps Engineer,Lynn Hancock\nContact Information:\n\n* Phone:...,True,Solid experience in machine learning and AI.,Help us build the next-generation products as ...
1108,Mobile App Developer,Charles Phillips\nMobile App Developer\n\nCont...,False,Insufficient system design expertise for senio...,We are looking for an experienced Mobile App D...


In [22]:
# Dataset 2
df_2.sample(10, random_state=0)

,resume_text,job_description_text,label
3069,"SummaryA results driven, detail-oriented data ...","Hello,Greetings from DevCare SolutionsI got an...",No Fit
1675,SummaryQuality-focused Data Entry Clerk experi...,Net2Source Inc. is an award-winning total work...,No Fit
6385,Career OverviewQuality focused SQL Data Analys...,The ideal candidate will be responsible for de...,No Fit
543,SummaryResults-oriented college graduate with ...,We are looking for an experienced engineer who...,No Fit
3213,Professional SummaryHighly motivated Sales Ass...,Kindly focus on the highlighted skill in the b...,Potential Fit
134,SummaryLogical Data Analyst skilled in require...,Job Purpose: Perform designated tasks in the a...,No Fit
5869,Experience11/2017toCurrentElectrical EngineerG...,"Hello, please look into the below mentioned ro...",Good Fit
1425,SummaryI am a Principal Electrical Engineer an...,I own a small landscaping business that needs ...,No Fit
1819,Professional SummaryHighly regarded self-start...,Skills - Cucumber BDD + Selenium UI Automation...,No Fit
7492,ProfileWith 11+ years of IT experience in Ent...,"Primary Location: Melbourne, Florida\nV-Soft C...",Potential Fit


In [23]:
# Dataset 3
df_3.sample(10, random_state=0)

,user_resume,user_prompt,assistant
18722,ACCOUNTANT III Senior Accountant Res...,What job category does this resume best fit?,This resume best fits the ACCOUNTANT category.
15478,INFORMATION TECHNOLOGY COORDINATOR Profe...,Rewrite and improve this resume for a INFORMAT...,Here's an improved version for a INFORMATION-T...
11686,Here is a professional resume for Brenda Horn:...,Please summarize the following resume:,Brenda Horn is a seasoned Cloud Architect with...
4691,here's a sample resume for steven frey applyin...,Please summarize the following resume:,Steven Frey is a skilled UI Engineer with over...
4653,"b'JOB SEEKER 128 Pleasant Street Minneapolis, ...",Please summarize the following resume:,The candidate holds a Master's in Education an...
13664,Elizabeth Richard Blockchain Developer Contact...,Please summarize the following resume:,Elizabeth Richard is a highly skilled blockcha...
13903,Gerald Morris Contact Information: * Address: ...,Please summarize the following resume:,Gerald Morris is an e-commerce professional wi...
4068,/\nNGUYỄN ĐÌNH DUẨN\nCHUYÊN VIÊN KINH DOANH\n...,Critique this resume and suggest improvements:,This resume could benefit from clearer formatt...
20442,Tấn Đạt Vũ Trương \nDEVOPS - 6 YEARS OF EXPERI...,Critique this resume and suggest improvements:,This resume could benefit from clearer formatt...
8380,INFORMATION TECHNOLOGY PROVISIONING TECHNICIAN...,What are the key skills and qualifications in ...,The key skills for this INFORMATION-TECHNOLOGY...


## Export

In [24]:
(DATASET_PATH / dataset_1_name).with_suffix('.csv')

WindowsPath('../data/dataset/AzharAli05_Resume_Screening_Dataset.csv')

In [25]:
# assume we stick with dataset 1
DATASET_PATH.mkdir(parents=True, exist_ok=True)

def write_dataset(name, df):
    df.to_csv(DATASET_PATH / (name + '.csv'), index=False)

write_dataset(dataset_1_name, df_1)